# qec-bench — Kaggle training run

Trains the per-distance neural decoders on Kaggle's free GPU tier
(30 h/week, quota independent of Colab). Enable **Settings -> Accelerator
-> GPU** and **Settings -> Internet -> On** before running.

**Resilient to disconnects.** Everything worth keeping is written under
`/kaggle/working/qecbench_backup/`, which Kaggle persists as this notebook's
output when a *Save & Run All (Commit)* finishes. To resume an interrupted or
extended run in a fresh session: **+ Add Input -> Your Work -> this notebook's
previous output**, then Run All — the restore cell below picks the backup up
from `/kaggle/input/`, finished stages are skipped, and training continues
from the last completed epoch.

In [ ]:
# Which dataset/model configs this run uses (see configs/ in the repo).
DATASET = "train_v2"   # 3M samples per distance — the data-scale run
TRAIN_CONFIG = "mlp_v3"


In [ ]:
!nvidia-smi -L

In [ ]:
import os
BACKUP = "/kaggle/working/qecbench_backup"
os.makedirs(f"{BACKUP}/weights", exist_ok=True)
print("backup dir:", BACKUP)

In [ ]:
import os
if not os.path.isdir("/kaggle/working/qec-bench"):
    !git clone https://github.com/Lucas-Maingi/qec-bench.git /kaggle/working/qec-bench
%cd /kaggle/working/qec-bench
!pip install -q -e ".[train]"


## Restore a previous session's progress (if its output is attached)

In [ ]:
import glob, os, shutil

for hit in glob.glob(f"/kaggle/input/*/qecbench_backup/{DATASET}"):
    if not os.path.isdir(f"{BACKUP}/{DATASET}"):
        print("restoring dataset from", hit)
        shutil.copytree(hit, f"{BACKUP}/{DATASET}")
for f in glob.glob("/kaggle/input/*/qecbench_backup/weights/*"):
    dest = f"{BACKUP}/weights/{os.path.basename(f)}"
    if not os.path.exists(dest):
        shutil.copy2(f, dest)
print("weights present:", sorted(os.listdir(f"{BACKUP}/weights")))

## 1/3 — Training dataset

Generated with Stim (CPU-bound) straight into the backup dir; skipped if a
previous session already produced it. Idempotent per (distance, error-rate)
block, so even an interrupted generation resumes.

In [ ]:
import os
if not os.path.exists(f"{BACKUP}/{DATASET}/meta.json"):
    !qecbench generate --config configs/datasets/$DATASET.yaml --out "$BACKUP"
    print("dataset generated")
else:
    print("dataset already present, skipping generation")

## 2/3 — Train the per-distance decoders

Checkpoints and JSONL metrics land in the backup dir, one write per epoch
(atomic writes — a dropped session can't corrupt the last good checkpoint).
Finished distances skip in milliseconds; an interrupted distance resumes from
its last completed epoch.

In [ ]:
!qecbench train --config configs/train/$TRAIN_CONFIG.yaml --data "$BACKUP/$DATASET" --out "$BACKUP/weights" --device cuda

## 3/3 — Hand back the weights

Commit the notebook (*Save & Run All*) so `/kaggle/working` persists, then
download the files in `qecbench_backup/weights/` from the Output tab (a few
MB). On the serving CPU:

```bash
# drop the checkpoints into weights_v3/ then
qecbench benchmark --dataset data/benchmark_v1     --decoders "pymatching,fusion_blossom,neural:weights,neural:weights_v3"     --out results/benchmark_v1.json
```

Latency numbers only count when measured on the hardware you ship inference
on — rerun the benchmark locally and commit that results file.

In [ ]:
import os
for f in sorted(os.listdir(f"{BACKUP}/weights")):
    print(f, os.path.getsize(f"{BACKUP}/weights/{f}") // 1024, "KB")